In [2]:
from pathlib import Path

readme = Path("README.md")

print("README exists:", readme.exists())
print("README size:", readme.stat().st_size, "bytes")

README exists: True
README size: 2172 bytes


In [4]:
from reader import read_all_csv

datasets = read_all_csv("olist")

print("Total CSV files:", len(datasets))

for name, records in datasets.items():
    print(name, ":", len(records), "records")
    

Total CSV files: 9
olist_customers_dataset : 99441 records
olist_geolocation_dataset : 1000163 records
olist_orders_dataset : 99441 records
olist_order_items_dataset : 112650 records
olist_order_payments_dataset : 103886 records
olist_order_reviews_dataset : 99224 records
olist_products_dataset : 32951 records
olist_sellers_dataset : 3095 records
product_category_name_translation : 71 records


In [5]:
from pathlib import Path

print(Path("config.json").read_text())


{
    "column_mapping": {
        "customer_id": "customer_id",
        "customer_unique_id": "unique_customer_id",
        "customer_zip_code_prefix": "postal_code",
        "customer_city": "city",
        "customer_state": "state",
        "order_id": "order_id",
        "order_status": "status",
        "order_purchase_timestamp": "purchase_date",
        "product_id": "product_id",
        "product_category_name": "category"
    }
}



In [7]:
import importlib
import reader

importlib.reload(reader)

datasets = reader.read_all_csv("olist")

print("Customer columns:")
print(datasets["olist_customers_dataset"][0].keys())

print("\nOrder columns:")
print(datasets["olist_orders_dataset"][0].keys())

Customer columns:
dict_keys(['customer_id', 'unique_customer_id', 'postal_code', 'city', 'state'])

Order columns:
dict_keys(['order_id', 'customer_id', 'status', 'purchase_date', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date'])


In [9]:
import pandas as pd
from validator import validate_dataframe

test_df = pd.DataFrame({
    "order_id": ["O1", "O2", "O3"],
    "customer_id": ["C1", None, "C3"],
    "status": ["delivered", "shipped", None]
})

valid_df, bad_df = validate_dataframe(test_df)

print("\nValid records:")
display(valid_df)

print("\nBad records:")
display(bad_df)

Total records: 3
Valid records: 1
Bad records: 2

Valid records:


,order_id,customer_id,status
0,O1,C1,delivered



Bad records:


,order_id,customer_id,status
1,O2,NaN,shipped
2,O3,C3,NaN


In [10]:
print(Path("reader.py").read_text())


import csv
import json
from pathlib import Path


def read_csv(path, column_mapping=None):
    with open(path, mode="r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        records = []

        for row in reader:
            if column_mapping:
                row = {
                    column_mapping.get(column, column): value
                    for column, value in row.items()
                }

            records.append(row)

        return records


def read_all_csv(directory, config_path="config.json"):
    with open(config_path, "r") as file:
        config = json.load(file)

    column_mapping = config.get("column_mapping", {})
    directory = Path(directory)
    datasets = {}

    for file_path in directory.glob("*.csv"):
        datasets[file_path.stem] = read_csv(
            file_path, column_mapping
        )

    return datasets



In [18]:
try:
    reader.read_csv("non_existing.csv")

except reader.DataReadError as e:
    print("Gracefully handled:", e)

Gracefully handled: Failed to read non_existing.csv: [Errno 2] No such file or directory: 'non_existing.csv'


In [19]:
from pathlib import Path

for file in Path("logs").glob("*.log"):
    print(file.name)

pipeline_20261123_150625.log
pipeline_20261123_150640.log
pipeline_20261123_152616.log
pipeline_20261123_152617.log
pipeline_20261123_152618.log
pipeline_20261123_152619.log
pipeline_20261123_152620.log
pipeline_20261123_154427.log
pipeline_20261123_154456.log


In [20]:
from pathlib import Path

log_files = sorted(Path("logs").glob("*.log"))

if log_files:
    latest_log = log_files[-1]

    print("Latest log:", latest_log.name)
    print(latest_log.read_text()[-2000:])
else:
    print("No log files found"
          )

Latest log: pipeline_20261123_154456.log
2026-11-23 15:44:56,255 | INFO | File already processed. Skipping pipeline.



In [1]:
import logging

logging.basicConfig(
    filename="logs/day5_test.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)


In [2]:
import importlib
import reader

importlib.reload(reader)

datasets = reader.read_all_csv("olist")

print("Datasets loaded:", len(datasets))

Datasets loaded: 9


In [3]:
from pathlib import Path

print(Path("logs/day5_test.log").read_text())

2026-11-25 09:39:55,359 | INFO | Configuration loaded successfully
2026-11-25 09:39:55,360 | INFO | Processing dataset: olist_customers_dataset.csv
2026-11-25 09:39:55,360 | INFO | Reading CSV: olist\olist_customers_dataset.csv
2026-11-25 09:39:55,791 | INFO | Successfully read 99441 records from olist\olist_customers_dataset.csv
2026-11-25 09:39:55,791 | INFO | Processing dataset: olist_geolocation_dataset.csv
2026-11-25 09:39:55,791 | INFO | Reading CSV: olist\olist_geolocation_dataset.csv
2026-11-25 09:39:59,623 | INFO | Successfully read 1000163 records from olist\olist_geolocation_dataset.csv
2026-11-25 09:39:59,623 | INFO | Processing dataset: olist_orders_dataset.csv
2026-11-25 09:39:59,623 | INFO | Reading CSV: olist\olist_orders_dataset.csv
2026-11-25 09:40:00,272 | INFO | Successfully read 99441 records from olist\olist_orders_dataset.csv
2026-11-25 09:40:00,272 | INFO | Processing dataset: olist_order_items_dataset.csv
2026-11-25 09:40:00,272 | INFO | Reading CSV: olist\olis

In [12]:

import hashlib
import json

manifest_path = output_dir / "manifest.json"


def get_file_hash(file_path):
    hasher = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


# Load existing manifest
if manifest_path.exists():
    with open(manifest_path, "r") as file:
        manifest = json.load(file)
else:
    manifest = {}


# Process all CSV files
for input_file in input_path.glob("*.csv"):

    current_hash = get_file_hash(input_file)

    output_csv = output_dir / f"{input_file.stem}_cleaned.csv"
    output_json = output_dir / f"{input_file.stem}_cleaned.json"

    # Skip only if unchanged and outputs already exist
    if (
        manifest.get(input_file.name) == current_hash
        and output_csv.exists()
        and output_json.exists()
    ):
        logger.info("File already processed. Skipping: %s", input_file.name)
        continue

    try:
        logger.info("Processing: %s", input_file.name)

        df = pd.read_csv(input_file)

        df.to_csv(output_csv, index=False)

        df.to_json(
            output_json,
            orient="records",
            indent=4
        )

        # Update manifest after successful processing
        manifest[input_file.name] = current_hash

        with open(manifest_path, "w") as file:
            json.dump(manifest, file, indent=4)

        logger.info("Successfully processed: %s", input_file.name)

    except (OSError, ValueError) as e:
        logger.exception("Failed to process: %s", input_file.name)


NameError: name 'output_dir' is not defined

In [9]:
import json
from pathlib import Path

notebook = json.loads(
    Path("day14.ipynb").read_text(encoding="utf-8")
)

print("".join(notebook["cells"][7]["source"]))

import hashlib
import json

manifest_path = output_dir / "manifest.json"

def get_file_hash(file_path):
    hasher = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


input_file = input_path / "olist_orders_dataset.csv"
current_hash = get_file_hash(input_file)

if manifest_path.exists():
    with open(manifest_path, "r") as file:
        manifest = json.load(file)
else:
    manifest = {}

if manifest.get(input_file.name) == current_hash:
    logger.info("File already processed. Skipping.")
else:
    df = pd.read_csv(input_file)

    df.to_csv(output_dir / "cleaned_data.csv", index=False)
    df.to_json(output_dir / "cleaned_data.json", orient="records", indent=4)

    manifest[input_file.name] = current_hash

    with open(manifest_path, "w") as file:
        json.dump(manifest, file, indent=4)

    logger.info("New or modified file processed successfully")


In [10]:
import json
from pathlib import Path

notebook = json.loads(
    Path("day14.ipynb").read_text(encoding="utf-8")
)

source = "".join(notebook["cells"][7]["source"])
lines = source.splitlines()

for start in range(0, len(lines), 15):
    print(f"\n--- Lines {start + 1} to {min(start + 15, len(lines))} ---")
    print("\n".join(lines[start:start + 15]))


--- Lines 1 to 15 ---
import hashlib
import json

manifest_path = output_dir / "manifest.json"

def get_file_hash(file_path):
    hasher = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            hasher.update(chunk)

    return hasher.hexdigest()



--- Lines 16 to 30 ---
input_file = input_path / "olist_orders_dataset.csv"
current_hash = get_file_hash(input_file)

if manifest_path.exists():
    with open(manifest_path, "r") as file:
        manifest = json.load(file)
else:
    manifest = {}

if manifest.get(input_file.name) == current_hash:
    logger.info("File already processed. Skipping.")
else:
    df = pd.read_csv(input_file)

    df.to_csv(output_dir / "cleaned_data.csv", index=False)

--- Lines 31 to 38 ---
    df.to_json(output_dir / "cleaned_data.json", orient="records", indent=4)

    manifest[input_file.name] = current_hash

    with open(manifest_path, "w") as file:
        json.dump(manifest, file, inden

In [11]:
for start in range(15, len(lines), 5):
    print(f"\n--- Lines {start + 1} to {min(start + 5, len(lines))} ---")
    print("\n".join(lines[start:start + 5]))
    


--- Lines 16 to 20 ---
input_file = input_path / "olist_orders_dataset.csv"
current_hash = get_file_hash(input_file)

if manifest_path.exists():
    with open(manifest_path, "r") as file:

--- Lines 21 to 25 ---
        manifest = json.load(file)
else:
    manifest = {}

if manifest.get(input_file.name) == current_hash:

--- Lines 26 to 30 ---
    logger.info("File already processed. Skipping.")
else:
    df = pd.read_csv(input_file)

    df.to_csv(output_dir / "cleaned_data.csv", index=False)

--- Lines 31 to 35 ---
    df.to_json(output_dir / "cleaned_data.json", orient="records", indent=4)

    manifest[input_file.name] = current_hash

    with open(manifest_path, "w") as file:

--- Lines 36 to 38 ---
        json.dump(manifest, file, indent=4)

    logger.info("New or modified file processed successfully")


In [14]:
from pathlib import Path

input_path = Path("olist")
output_dir = Path("output")

output_dir.mkdir(parents=True, exist_ok=True)

print("Input directory:", input_path)
print("Output directory:", output_dir)
print("Input exists:", input_path.exists())

Input directory: olist
Output directory: output
Input exists: True


In [16]:
import json

with open("output/manifest.json", "r") as file:
    manifest = json.load(file)

print("Total files tracked:", len(manifest))
print(list(manifest.keys()))

Total files tracked: 1
['olist_orders_dataset.csv']


In [18]:
import logging
from pathlib import Path

Path("logs").mkdir(exist_ok=True)

logger = logging.getLogger("OlistPipeline")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.FileHandler("logs/day15_test.log", encoding="utf-8")
    handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    )
    logger.addHandler(handler)

print("Logger initialized successfully")

Logger initialized successfully


In [23]:

import hashlib
import json

manifest_path = output_dir / "manifest.json"


# Generate MD5 hash
def get_file_hash(file_path):
    hasher = hashlib.md5()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


# Load existing manifest
if manifest_path.exists():
    with open(manifest_path, "r") as file:
        manifest = json.load(file)
else:
    manifest = {}


# Process all CSV files
for input_file in sorted(input_path.glob("*.csv")):

    current_hash = get_file_hash(input_file)

    output_csv = output_dir / f"{input_file.stem}_cleaned.csv"
    output_json = output_dir / f"{input_file.stem}_cleaned.json"

    # Check if file is already processed
    if (
    manifest.get(input_file.name) == current_hash
    and output_csv.exists()
    and output_json.exists()
):
        message = f"File already processed. Skipping: {input_file.name}"
        logger.info(message)
        print(message)
        continue

    try:
        logger.info("Processing dataset: %s", input_file.name)

        df = pd.read_csv(input_file)

        # Save CSV
        df.to_csv(output_csv, index=False)

        # Save JSON
        df.to_json(
            output_json,
            orient="records",
            indent=4
        )

        # Update manifest
        manifest[input_file.name] = current_hash

        with open(manifest_path, "w") as file:
            json.dump(manifest, file, indent=4)

        logger.info(
            "Successfully processed: %s",
            input_file.name
        )

    except (OSError, ValueError):
        logger.exception(
            "Failed to process: %s",
            input_file.name
        )

print("Pipeline execution completed.")
print("Files tracked:", len(manifest))


File already processed. Skipping: olist_customers_dataset.csv
File already processed. Skipping: olist_geolocation_dataset.csv
File already processed. Skipping: olist_order_items_dataset.csv
File already processed. Skipping: olist_order_payments_dataset.csv
File already processed. Skipping: olist_order_reviews_dataset.csv
File already processed. Skipping: olist_orders_dataset.csv
File already processed. Skipping: olist_products_dataset.csv
File already processed. Skipping: olist_sellers_dataset.csv
File already processed. Skipping: product_category_name_translation.csv
Pipeline execution completed.
Files tracked: 9


In [24]:
!python -m pytest test_functions.py -v

============================= test session starts =============================
platform win32 -- Python 3.12.9, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\Santhosh Kumar\AppData\Local\Programs\Python\Python312\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Santhosh Kumar\Desktop\python
plugins: anyio-4.16.0, platformdirs-4.14.5
collecting ... collected 5 items

test_functions.py::test_read_csv PASSED                                  [ 20%]
test_functions.py::test_to_snake_case PASSED                             [ 40%]
test_functions.py::test_calculate_order_totals PASSED                    [ 60%]
test_functions.py::test_read_csv_invalid_file PASSED                     [ 80%]
test_functions.py::test_to_snake_case_empty_list PASSED                  [100%]

============================== 5 passed in 1.27s ==============================


In [25]:
!git status

On branch feature
Your branch is up to date with 'origin/feature'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   __pycache__/reader.cpython-312.pyc
	modified:   __pycache__/test_functions.cpython-312-pytest-9.2.0.pyc
	modified:   __pycache__/transformer.cpython-312.pyc
	modified:   __pycache__/validator.cpython-312.pyc
	modified:   day11.ipynb
	modified:   day12.ipynb
	modified:   reader.py
	modified:   test_functions.py
	modified:   transformer.py
	modified:   validator.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README.MD
	bad_records.csv
	config.json
	config.yaml
	day13.ipynb
	day14.ipynb
	day15.ipynb
	logs/
	output/

no changes added to commit (use "git add" and/or "git commit -a")
